# NB13 PEST-R: rebuild the audit package and make the files to send (CPU only, about 20-30 min)
Settings: Accelerator = None, Internet = ON. Run top to bottom. The sampling is fixed (seed 20261106), so this rebuilds exactly the same 100 items as before; use the key file from THIS run.
Download from the right-hand Output panel: /kaggle/working/audit_send/ (two reviewer zips + the private key).

## 1. Restore from Hugging Face (15-20 min, asks for a read token with getpass)

In [ ]:
# R1: restore everything from Hugging Face into /kaggle/working (about 10-15 min the first time in a session)
import os, glob, subprocess
from getpass import getpass
REPO = "Uzairezra/pest-backup"
if not os.path.exists("/kaggle/working/pest_v4/groups_final.json"):
    from huggingface_hub import list_repo_files, hf_hub_download
    tok = getpass("Hugging Face token (read is enough): ")
    zips = sorted([f for f in list_repo_files(REPO, repo_type="dataset", token=tok) if f.endswith(".zip")], key=lambda f: (f != "pest_v4_backup.zip", f))
    print("zips in repo:", zips)
    for f in zips:
        hf_hub_download(REPO, f, repo_type="dataset", local_dir="/kaggle/working", token=tok)
        subprocess.run(["unzip", "-q", "-o", f"/kaggle/working/{f}", "-d", "/kaggle/working"], check=True); print("restored", f)
import sys; sys.path.insert(0, "/kaggle/working")
from pest_common import *
Gf, M, D = load_matched()
print("FINAL matched groups:", len(M), "| DEV:", len(D), "| prediction files:", len(glob.glob(P("preds", "*.json"))))
print("COCO reachable:", os.path.exists(M[0]["files"]["original"]), "| natural-image path ok:", os.path.exists(jload(P("nat.json"))[0]["path"]))


## 2. Write the builder and the QA script

In [ ]:
%%writefile /kaggle/working/n11_audit_package.py
# >>> N11: HUMAN EDIT-VALIDITY AUDIT PACKAGE (CPU only, ~2-5 min). Builds the package only. NO audit has been done by this code.
# Reuses the edit images that already exist from the test set (no new editing, no model, no GPU). Writes only into a NEW folder.
# Run on Kaggle after restoring pest_v4 (same as the other analysis cells). Set EDITOR to "sd" (main audit: the held-out test editor the paper
# reports as 'clean'), then optionally run again with "crude" (and "lama"/"medium" if wanted). Output is one folder per editor.
#
# Design (independent of model tuning):
#   * Sample N_GROUPS groups at random from the 235 matched TEST groups with a fixed seed. Predictions, margins and results are never read.
#     At most one group per COCO source image. Sampling is not stratified by anything the models did.
#   * Each item shows: ORIGINAL | EDIT 1 | EDIT 2. Which edit is the removal and which is the placebo is randomised per item and HIDDEN from reviewers
#     (the key is a separate file). Reviewers therefore answer the same questions about both edits, so (a) and (b) are judged blind to intent.
#   * Two separate workbooks (reviewer_A.xlsx, reviewer_B.xlsx) with different row orders and coded item ids: reviewers work independently.
#   * Questions per edit: is the target object still visible?  are there visible editing artifacts / an obvious failure?  Plus: is the object in the original?
#   * Unblinding (done only after both files are returned) maps the answers to the three audit questions:
#       (a) target removed       = removal edit judged "Not visible"
#       (b) target preserved     = placebo edit judged "Clearly visible"
#       (c) artifacts / failure  = "Minor" / "Obvious failure" on either edit
import os, sys, json, csv, random, shutil, math, collections
sys.path.insert(0, "/kaggle/working")
from PIL import Image, ImageDraw, ImageFont

EDITOR = os.environ.get("AUDIT_EDITOR", "sd")             # sd | lama | crude | medium
N_GROUPS = int(os.environ.get("AUDIT_N", 100))
SAMPLE_SEED = 20261106                                    # fixed, written to the Instructions sheet
OUT = os.environ.get("AUDIT_OUT", f"/kaggle/working/pest_audit_{EDITOR}")
PANEL_W = 420                                             # montage panel width (full-resolution PNGs are copied separately)

OPT_VIS = ["Clearly visible", "Partly visible", "Not visible", "Cannot tell"]
OPT_ART = ["None", "Minor", "Obvious failure"]
OPT_YN = ["Yes", "No", "Cannot tell"]

def edit_paths(g, editor, P=None, cpath=None, mpath=None):
    """(removal_path, placebo_path) of the EXISTING edit files for this editor."""
    if editor in ("sd", "lama"): return g["files"][f"{editor}_remove"], g["files"][f"{editor}_placebo"]
    if editor == "crude": return cpath(g["gid"], "remove"), cpath(g["gid"], "placebo")
    if editor == "medium": return mpath(g["gid"], "remove"), mpath(g["gid"], "placebo")
    raise SystemExit("unknown editor")

def sample_groups(M, n, seed):
    rnd = random.Random(seed); order = sorted(M, key=lambda g: g["gid"]); rnd.shuffle(order)
    seen, out = set(), []
    for g in order:
        if g["img_id"] in seen: continue
        seen.add(g["img_id"]); out.append(g)
        if len(out) == n: break
    return out

def font(sz):
    for f in ("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "DejaVuSans-Bold.ttf"):
        try: return ImageFont.truetype(f, sz)
        except Exception: pass
    return ImageFont.load_default()

def montage(paths, labels, out_path):
    ims = []
    for p in paths:
        im = Image.open(p).convert("RGB"); h = int(im.height * PANEL_W / im.width); ims.append(im.resize((PANEL_W, h), Image.LANCZOS))
    H = max(i.height for i in ims) + 30; W = PANEL_W * len(ims) + 8 * (len(ims) - 1)
    M = Image.new("RGB", (W, H), (255, 255, 255)); d = ImageDraw.Draw(M); f = font(18)
    for k, (im, lab) in enumerate(zip(ims, labels)):
        x = k * (PANEL_W + 8); M.paste(im, (x, 30)); d.text((x + 6, 5), lab, fill=(0, 0, 0), font=f)
    M.save(out_path, "JPEG", quality=88); return M.size

def build(M, editor, out, n, seed, P=None, cpath=None, mpath=None, qtext=None):
    S = sample_groups(M, n, seed)
    if len(S) < n: print(f"WARNING: only {len(S)} usable groups (asked {n})")
    for d in ("images", "montages"): os.makedirs(os.path.join(out, d), exist_ok=True)
    rnd = random.Random(seed + 1); codes = [f"A{i+1:03d}" for i in range(len(S))]; rnd.shuffle(codes)   # codes carry no information about the group
    key, items = [], []
    for g, code in zip(S, codes):
        rem, pla = edit_paths(g, editor, P, cpath, mpath)
        for p in (g["files"]["original"], rem, pla):
            if not os.path.exists(p): raise SystemExit(f"missing existing image file: {p}")
        removal_is_1 = rnd.random() < 0.5
        e1, e2 = (rem, pla) if removal_is_1 else (pla, rem)
        names = {"original": g["files"]["original"], "edit1": e1, "edit2": e2}
        for k, p in names.items(): shutil.copyfile(p, os.path.join(out, "images", f"{code}_{k}.png"))      # lossless copies, no re-encoding
        montage([names["original"], e1, e2], ["ORIGINAL", "EDIT 1", "EDIT 2"], os.path.join(out, "montages", f"{code}.jpg"))
        obj = g["cat"].replace("_", " ")
        items.append(dict(code=code, obj=obj, question=(qtext(g["cat"]) if qtext else f"Is there a {obj} in the image?")))
        key.append(dict(code=code, gid=g["gid"], img_id=g["img_id"], category=g["cat"], editor=editor, edit1_is="removal" if removal_is_1 else "placebo",
                        edit2_is="placebo" if removal_is_1 else "removal", original=g["files"]["original"], removal=rem, placebo=pla,
                        placebo_area=g.get("placebo_area"), remove_area=g.get("remove_area")))
    with open(os.path.join(out, "audit_KEY_do_not_share_with_reviewers.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(key[0])); w.writeheader(); w.writerows(key)
    return items, key

INSTR = [
 ("PEST-R edit-validity audit  -  reviewer instructions", ""),
 ("Status", "THIS IS A BLANK AUDIT FORM. No human audit has been completed by the person or system that built it. Results exist only after two reviewers return filled forms."),
 ("Purpose", "A research paper edits COCO photographs with an image editor to (i) REMOVE a named object and (ii) make a control edit of the same size and shape somewhere ELSE that should leave the named object visible. You check whether the edits worked. Please do not guess what the research hypothesis is."),
 ("Data disclosure", "All images are taken from the held-out TEST set of the study (not from training data), and all edits are the pre-existing test edits (editor: %s). The groups were sampled uniformly at random with fixed seed %d, at most one per source photo, without looking at any model output. Reviewers must not use any model to help judge."),
 ("What you see", "For every row: the ORIGINAL photo and two edited versions, EDIT 1 and EDIT 2 (shown in one strip in the 'Image' column; full-resolution PNG files are in the images/ folder under the names given in the 'Full-size files' column). One of the two edits is a removal and the other is a control edit - you are NOT told which, and their order is random. Judge each edit on its own."),
 ("Q1  Target visible in the ORIGINAL?", "Yes / No / Cannot tell. Is the named object (column 'Target object') present in the original photo? (A sanity check on the data.)"),
 ("Q2  Target visible in EDIT 1 / EDIT 2", "Clearly visible = you can recognise the named object at a glance.  Partly visible = only a part, or a remnant/ghost that a person could still recognise.  Not visible = no recognisable trace of the object remains.  Cannot tell = the image is too unclear to decide. Use only what you can see; do not infer from the edit's shape."),
 ("Q3  Artifacts in EDIT 1 / EDIT 2", "None = the edit looks natural.  Minor = a visible smear, texture, colour or edge mismatch, but the image is otherwise sensible.  Obvious failure = a flat patch, large distortion, an object that was added, a duplicated object, or anything that makes the edit clearly unusable."),
 ("Notes", "Free text. Write anything unusual (e.g. 'another object of the same kind is still visible', 'wrong object removed', 'image broken'). Leave blank if nothing."),
 ("Independence", "Work alone. Do not discuss rows with the other reviewer until both files are submitted. Fill ONLY the yellow columns. Do not reorder, delete or rename rows. Save as .xlsx and return the file."),
 ("Time", "About 1 to 2 minutes per row; plan about 2-3 hours for 100 rows. It is fine to work in several sittings."),
]

def write_workbooks(items, key, out, editor, seed):
    from openpyxl import Workbook
    from openpyxl.drawing.image import Image as XImg
    from openpyxl.worksheet.datavalidation import DataValidation
    from openpyxl.styles import PatternFill, Font, Alignment
    yellow = PatternFill("solid", fgColor="FFF2A8"); head = PatternFill("solid", fgColor="D9E2F3")
    for who, shuf in (("A", seed + 11), ("B", seed + 22)):
        order = list(range(len(items))); random.Random(shuf).shuffle(order)
        wb = Workbook(); ws0 = wb.active; ws0.title = "Instructions"; ws0.column_dimensions["A"].width = 34; ws0.column_dimensions["B"].width = 130
        for r, (a, b) in enumerate(INSTR, 1):
            b = b % (editor, seed) if "%s" in b else b
            ws0.cell(r, 1, a).font = Font(bold=True); c = ws0.cell(r, 2, b); c.alignment = Alignment(wrap_text=True, vertical="top"); ws0.cell(r, 1).alignment = Alignment(wrap_text=True, vertical="top")
        ws0.cell(len(INSTR) + 2, 1, "Reviewer").font = Font(bold=True); ws0.cell(len(INSTR) + 2, 2, "Reviewer %s  -  enter your name/initials here: ______" % who)
        ws = wb.create_sheet("Audit")
        cols = ["Row", "Item code", "Target object", "Image (ORIGINAL | EDIT 1 | EDIT 2)", "Full-size files", "Q1 Target in ORIGINAL?", "Q2 Edit 1: target visible?", "Q3 Edit 1: artifacts?",
                "Q2 Edit 2: target visible?", "Q3 Edit 2: artifacts?", "Notes"]
        widths = [6, 10, 16, 95, 34, 16, 20, 18, 20, 18, 40]
        for j, (c, w) in enumerate(zip(cols, widths), 1):
            cell = ws.cell(1, j, c); cell.font = Font(bold=True); cell.fill = head; cell.alignment = Alignment(wrap_text=True, vertical="center"); ws.column_dimensions[chr(64 + j)].width = w
        dvs = {"F": OPT_YN, "G": OPT_VIS, "H": OPT_ART, "I": OPT_VIS, "J": OPT_ART}
        for col, opts in dvs.items():
            dv = DataValidation(type="list", formula1='"' + ",".join(opts) + '"', allow_blank=True); ws.add_data_validation(dv); dv.add(f"{col}2:{col}{len(items) + 1}")
        for r, i in enumerate(order, 2):
            it = items[i]; ws.row_dimensions[r].height = 175
            ws.cell(r, 1, r - 1); ws.cell(r, 2, it["code"]); ws.cell(r, 3, it["obj"])
            ws.cell(r, 5, f"images/{it['code']}_original.png\nimages/{it['code']}_edit1.png\nimages/{it['code']}_edit2.png")
            for j in (3, 5): ws.cell(r, j).alignment = Alignment(wrap_text=True, vertical="top")
            img = XImg(os.path.join(out, "montages", it["code"] + ".jpg")); sc = 650 / img.width; img.width, img.height = int(img.width * sc), int(img.height * sc); ws.add_image(img, f"D{r}")
            for j in range(6, 12): ws.cell(r, j).fill = yellow; ws.cell(r, j).alignment = Alignment(wrap_text=True, vertical="top")
        ws.freeze_panes = "D2"
        wb.save(os.path.join(out, f"reviewer_{who}.xlsx"))

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 3
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d; return p, c - h, c + h

def kappa(a, b):
    labs = sorted(set(a) | set(b)); n = len(a)
    if n == 0: return float("nan")
    po = sum(x == y for x, y in zip(a, b)) / n; pe = sum((a.count(l) / n) * (b.count(l) / n) for l in labs)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

def analyse(out, fileA=None, fileB=None):
    """RUN ONLY AFTER both reviewers returned their filled forms. Unblinds with the key, reports rates, 95% Wilson CIs, agreement."""
    from openpyxl import load_workbook
    fileA = fileA or os.path.join(out, "reviewer_A_FILLED.xlsx"); fileB = fileB or os.path.join(out, "reviewer_B_FILLED.xlsx")
    key = {r["code"]: r for r in csv.DictReader(open(os.path.join(out, "audit_KEY_do_not_share_with_reviewers.csv")))}
    def read(f):
        ws = load_workbook(f)["Audit"]; R = {}
        for row in ws.iter_rows(min_row=2, values_only=True):
            if row[1]: R[row[1]] = dict(orig=row[5], v1=row[6], a1=row[7], v2=row[8], a2=row[9], note=row[10])
        return R
    RA, RB = read(fileA), read(fileB); lines = []
    def pr(s): print(s); lines.append(s)
    def per_item(R):
        o = {}
        for c, r in R.items():
            k = key[c]; rem1 = k["edit1_is"] == "removal"
            vr, ar = (r["v1"], r["a1"]) if rem1 else (r["v2"], r["a2"]); vp, ap = (r["v2"], r["a2"]) if rem1 else (r["v1"], r["a1"])
            nn = lambda x: x is not None and str(x).strip() != ""
            o[c] = dict(removed=(vr == "Not visible") if nn(vr) else None, preserved=(vp == "Clearly visible") if nn(vp) else None,
                        art_rem=(ar in ("Minor", "Obvious failure")) if nn(ar) else None, art_pla=(ap in ("Minor", "Obvious failure")) if nn(ap) else None,
                        fail_rem=(ar == "Obvious failure") if nn(ar) else None, fail_pla=(ap == "Obvious failure") if nn(ap) else None)
        return o
    IA, IB = per_item(RA), per_item(RB); codes = sorted(set(IA) & set(IB))
    pr(f"Editor: {next(iter(key.values()))['editor']}   items in both files: {len(codes)}   (completeness is counted separately for every metric)")
    for name, f in (("(a) target removed in the removal edit", "removed"), ("(b) target preserved in the placebo edit", "preserved"), ("(c1) any artifact in the removal edit", "art_rem"),
                    ("(c2) any artifact in the placebo edit", "art_pla"), ("(c3) obvious failure, removal edit", "fail_rem"), ("(c4) obvious failure, placebo edit", "fail_pla")):
        for who, I in (("A", IA), ("B", IB)):
            ok = [I[c][f] for c in codes if I[c][f] is not None]; p, lo, hi = wilson(sum(ok), len(ok))
            pr(f"  {name:<44} reviewer {who}: {sum(ok)}/{len(ok)} = {p:.3f} [{lo:.3f},{hi:.3f}]   (answered {len(ok)}/{len(codes)})")
        pair = [c for c in codes if IA[c][f] is not None and IB[c][f] is not None]
        both = [IA[c][f] and IB[c][f] for c in pair]; p, lo, hi = wilson(sum(both), len(pair))
        k = kappa([IA[c][f] for c in pair], [IB[c][f] for c in pair])
        pr(f"  {'':<44} BOTH yes: {sum(both)}/{len(pair)} = {p:.3f} [{lo:.3f},{hi:.3f}]   kappa = {k:.2f}   (rated by both: {len(pair)}/{len(codes)})")
    pr("Disagreements and notes to adjudicate (item code: A vs B):")
    for c in codes:
        for f in ("removed", "preserved"):
            if IA[c][f] is not None and IB[c][f] is not None and IA[c][f] != IB[c][f]: pr(f"  {c} {f}: A={IA[c][f]} B={IB[c][f]}")
    open(os.path.join(out, "audit_analysis.txt"), "w").write("\n".join(lines)); return lines

if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "analyse":
        analyse(OUT); raise SystemExit
    from pest_common import *
    try:
        import pest_study as PS
    except Exception as e:
        PS = None; print('pest_study not importable (fine for editor sd/lama):', e)
    Gf, M, D = load_matched()
    if os.path.exists(OUT): raise SystemExit(f"{OUT} already exists - refusing to overwrite. Set AUDIT_OUT to a new folder.")
    items, key = build(M, EDITOR, OUT, N_GROUPS, SAMPLE_SEED, cpath=getattr(PS, 'cpath', None), mpath=getattr(PS, 'mpath', None), qtext=None)
    write_workbooks(items, key, OUT, EDITOR, SAMPLE_SEED)
    cnt = collections.Counter(k["category"] for k in key)
    print(f"built {len(items)} items in {OUT}"); print("category mix:", dict(cnt.most_common()))
    print("files: reviewer_A.xlsx, reviewer_B.xlsx (give one to each reviewer, with the images/ folder),\n       audit_KEY_do_not_share_with_reviewers.csv (keep yourself)")


In [ ]:
%%writefile /kaggle/working/audit_qa.py
#!/usr/bin/env python3
"""QA of the audit package BEFORE it goes to reviewers. Read-only except for one new file (qa_contact_sheet.jpg).
   python audit_qa.py  PATH_TO_pest_audit_sd      (folder that holds reviewer_A.xlsx, reviewer_B.xlsx, images/, montages/, the KEY csv)
Needs: openpyxl, pillow. Prints PASS/FAIL lines; open qa_contact_sheet.jpg yourself and look at it."""
import os, sys, csv, random, re
from openpyxl import load_workbook
from PIL import Image
root = sys.argv[1] if len(sys.argv) > 1 else "/kaggle/working/pest_audit_sd"
ok = True
def chk(cond, msg):
    global ok; ok &= bool(cond); print(("PASS  " if cond else "FAIL  ") + msg)
key = list(csv.DictReader(open(os.path.join(root, "audit_KEY_do_not_share_with_reviewers.csv"))))
codes = [k["code"] for k in key]; n = len(codes)
chk(n >= int(os.environ.get("AUDIT_N", 100)) * 0.9, f"key has {n} items (expected about {os.environ.get('AUDIT_N', 100)}); codes unique: {len(set(codes)) == n}")
chk(len({k['img_id'] for k in key}) == n, "one group per source photo")
chk(all(k["edit1_is"] != k["edit2_is"] for k in key), "every item has one removal and one placebo edit")
r1 = sum(k["edit1_is"] == "removal" for k in key); chk(abs(r1 - n / 2) <= 0.2 * n + 3, f"removal is Edit 1 in {r1}/{n} items (should be near half, so position gives nothing away)")
miss = [c for c in codes for s in ("original", "edit1", "edit2") if not os.path.exists(os.path.join(root, "images", f"{c}_{s}.png"))]
chk(not miss, f"all {3*n} full-size PNGs present" + ("" if not miss else f" (missing e.g. {miss[:3]})"))
chk(all(os.path.exists(os.path.join(root, "montages", c + ".jpg")) for c in codes), "all montages present")
sizes = [Image.open(os.path.join(root, "images", f"{c}_original.png")).size for c in codes[:20]]
chk(all(w >= 200 and h >= 200 for w, h in sizes), f"images open and look full size (e.g. {sizes[0]})")
secret = [k["gid"] for k in key] + [k["removal"] for k in key[:5]]
orders = []
for who in "AB":
    p = os.path.join(root, f"reviewer_{who}.xlsx"); wb = load_workbook(p)
    chk(set(wb.sheetnames) == {"Instructions", "Audit"}, f"reviewer_{who}: sheets {wb.sheetnames}")
    ws = wb["Audit"]; rows = [r for r in ws.iter_rows(min_row=2, values_only=True) if r[1]]
    chk(len(rows) == n, f"reviewer_{who}: {len(rows)} item rows; embedded images: {len(ws._images)}")
    chk(len(ws._images) == n, f"reviewer_{who}: one picture per row")
    dv = [str(d.sqref) for d in ws.data_validations.dataValidation]
    chk(len(dv) == 5, f"reviewer_{who}: 5 dropdown columns {dv}")
    blank = all(all(c is None for c in r[5:11]) for r in rows); chk(blank, f"reviewer_{who}: answer columns are blank")
    text = " ".join(str(c) for r in rows for c in r[:5] if c is not None) + " ".join(str(c.value) for row in wb["Instructions"].iter_rows() for c in row if c.value)
    leak = [w for w in ("removal", "placebo") if re.search(w, " ".join(str(c) for r in rows for c in r[:5] if c is not None), re.I)]
    chk(not leak, f"reviewer_{who}: item rows do not say which edit is which (found words: {leak or 'none'})")
    chk(not any(g in text for g in secret), f"reviewer_{who}: no group ids or source paths in the workbook")
    orders.append([r[1] for r in rows])
chk(orders[0] != orders[1], "the two reviewers see different row orders")
chk("audit_KEY" not in " ".join(os.listdir(os.path.join(root, "images"))), "no key file inside images/")
print("\nFilenames inside images/ look like:", sorted(os.listdir(os.path.join(root, "images")))[:3], "(code + original/edit1/edit2 only - good)")
# contact sheet: 6 random items, enlarge to eyeball the edits
random.seed(1); pick = random.sample(codes, 6); ims = [Image.open(os.path.join(root, "montages", c + ".jpg")) for c in pick]
W = max(i.width for i in ims); H = sum(i.height for i in ims); sheet = Image.new("RGB", (W, H), "white"); y = 0
for i in ims: sheet.paste(i, (0, y)); y += i.height
sheet.save(os.path.join(root, "qa_contact_sheet.jpg"), quality=88); print("wrote", os.path.join(root, "qa_contact_sheet.jpg"), "- OPEN IT and check: object visibly gone in one edit, still visible in the other, no broken images")
print("\nRESULT:", "ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED - do not send to reviewers yet")


## 3. Build the package (3-6 min)

In [ ]:
!cd /kaggle/working && AUDIT_EDITOR=sd python n11_audit_package.py

## 4. QA: all lines must say PASS. Then LOOK at the contact sheet below.

In [ ]:
!cd /kaggle/working && python audit_qa.py /kaggle/working/pest_audit_sd

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/pest_audit_sd/qa_contact_sheet.jpg"))

## 5. Make the zips to send (1-2 min)

In [ ]:
# Build one zip per reviewer (xlsx + full-size images + a short note) and keep the key file separate.
import os, zipfile, shutil
ROOT = "/kaggle/working/pest_audit_sd"; OUTD = "/kaggle/working/audit_send"; os.makedirs(OUTD, exist_ok=True)
NOTE = """PEST-R edit audit: how to do it
1. Unzip this folder. Keep reviewer_X.xlsx and the images/ folder side by side.
2. Open the xlsx (Excel or LibreOffice). Read the Instructions sheet, then fill ONLY the yellow cells on the Audit sheet using the dropdowns.
3. Work alone; do not discuss rows with anyone, including the other reviewer, until both of you have returned your file.
4. Do not use any AI model or automated tool to help judge the images.
5. Save the file under the same name with _FILLED added (for example reviewer_A_FILLED.xlsx) and send it back.
About 2-3 hours for 100 rows. You may work in several sittings."""
for who in "AB":
    z = f"{OUTD}/reviewer_{who}_package.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_STORED) as Z:                     # PNGs are already compressed: store, do not recompress
        Z.write(f"{ROOT}/reviewer_{who}.xlsx", f"reviewer_{who}.xlsx"); Z.writestr("READ_ME_FIRST.txt", NOTE)
        for f in sorted(os.listdir(f"{ROOT}/images")): Z.write(f"{ROOT}/images/{f}", f"images/{f}")
    print(z, round(os.path.getsize(z) / 1e6), "MB")
shutil.copy(f"{ROOT}/audit_KEY_do_not_share_with_reviewers.csv", f"{OUTD}/audit_KEY_PRIVATE.csv")
print("\nSEND reviewer_A_package.zip to reviewer A and reviewer_B_package.zip to reviewer B. KEEP audit_KEY_PRIVATE.csv yourself.")
